## Hyperparameter Tuning for GRU

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q optuna
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [ ]:
import json

import numpy as np
import optuna

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import GRUPredictor, HORIZONS

optuna.logging.set_verbosity(optuna.logging.WARNING)

MODEL_NAME = 'GRU-tuned'

TRIAL_SEEDS = [7, 21, 35, 49, 63]
N_TRIALS = 40

SEARCH_SEED = 0  # seeds Optuna's RandomSampler itself

FINAL_SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
DEFAULT_PARAMS = dict(hidden_size=128, num_layers=2, dropout=0.2, lr=1e-3, weight_decay=1e-4,
                       batch_size=256)

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '2_best_cgm_encoder'
RESULTS_DIR = EXPERIMENT_DIR / 'results'
TRIALS_CSV_PATH = RESULTS_DIR / 'gru_tuning_trials.csv'
FINAL_SUMMARY_PATH = RESULTS_DIR / 'gru_tuning_final_summary.json'
DEFAULT_SUMMARY_PATH = PROJECT_ROOT / 'experiments' / '1_baseline' / 'results' / 'gru' / 'gru_stability_summary.json'

In [ ]:
stats = load_cgm_stats()
val_loader = load_tensors('val', batch_size=256, shuffle=False)
test_loader = load_tensors('test', batch_size=256, shuffle=False)

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))


print(f'number of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]


In [ ]:
def objective(trial):
    hidden_size = trial.suggest_categorical('hidden_size', [64, 128, 256])
    num_layers = trial.suggest_categorical('num_layers', [1, 2, 3])
    dropout = trial.suggest_categorical('dropout', [0.1, 0.2, 0.3])
    lr = trial.suggest_categorical('lr', [1e-3, 1e-4, 1e-5])
    weight_decay = trial.suggest_categorical('weight_decay', [1e-4, 1e-5, 1e-6])
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256, 512])

    print(f"\n{'-' * 16}trial={trial.number}{'-' * 16}")
    print(f"hidden_size={hidden_size}  num_layers={num_layers}  dropout={dropout:.3f}  "
          f"lr={lr:.2e}  weight_decay={weight_decay:.2e}  batch_size={batch_size}")

    trial_train_loader = load_tensors('train', batch_size=batch_size)

    seed_val_losses = []
    for s in TRIAL_SEEDS:
        train.set_seed(s)
        model = GRUPredictor(hidden_size=hidden_size, num_layers=num_layers, dropout=dropout)
        model, history = train.train_model(
            model, trial_train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
            lr=lr, weight_decay=weight_decay,
            checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / f'trial{trial.number}_seed{s}',
            verbose=False, progress_bar=True, progress_desc=f'trial={trial.number} seed={s}',
        )
        best_val_loss = min(history['val_loss'])
        seed_val_losses.append(best_val_loss)
        print(f"  trial={trial.number}  seed={s}  best_val_loss={best_val_loss:.4f}")

    mean_val_loss = float(np.mean(seed_val_losses))
    trial.set_user_attr('seed_val_losses', seed_val_losses)
    print(f"trial={trial.number:>3}  mean_val_loss over {len(TRIAL_SEEDS)} seeds = "
          f"{mean_val_loss:.4f}")
    return mean_val_loss

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sampler = optuna.samplers.RandomSampler(seed=SEARCH_SEED)
study = optuna.create_study(direction='minimize', sampler=sampler)
study.enqueue_trial(DEFAULT_PARAMS)
study.optimize(objective, n_trials=N_TRIALS)

print(f"\n=== best trial: #{study.best_trial.number} ===")
print('best_params:', study.best_params)
print(f'best mean_val_loss (over {len(TRIAL_SEEDS)} seeds):', study.best_value)

trials_df = study.trials_dataframe()
trials_df.to_csv(TRIALS_CSV_PATH, index=False)
print(f'\nsaved all {N_TRIALS} trials to {TRIALS_CSV_PATH}')

# Where did the explicitly-enqueued default config rank among all N_TRIALS, by the same criterion?
completed = sorted((t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE),
                    key=lambda t: t.value)
default_rank = next(i for i, t in enumerate(completed, start=1) if t.params == DEFAULT_PARAMS)
print(f'default config ranked #{default_rank} of {len(completed)} by mean_val_loss')


----------------trial=0----------------
hidden_size=128  num_layers=2  dropout=0.200  lr=1.00e-03  weight_decay=1.00e-04  batch_size=256
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 157,957


trial=0 seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

  trial=0  seed=7  best_val_loss=0.3258
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 157,957


trial=0 seed=21:   0%|          | 0/50 [00:00<?, ?it/s]

  trial=0  seed=21  best_val_loss=0.3286
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 157,957


...

  trial=39  seed=35  best_val_loss=0.3284
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 65,029


trial=39 seed=49:   0%|          | 0/50 [00:00<?, ?it/s]

  trial=39  seed=49  best_val_loss=0.3307
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 65,029


trial=39 seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

  trial=39  seed=63  best_val_loss=0.3281
trial= 39  mean_val_loss over 5 seeds = 0.3292

=== best trial: #24 ===
best_params: {'hidden_size': 256, 'num_layers': 3, 'dropout': 0.2, 'lr': 0.0001, 'weight_decay': 1e-05, 'batch_size': 256}
best mean_val_loss (over 5 seeds): 0.32593515515327454

saved all 40 trials to /content/drive/MyDrive/dissertation/experiments/2_best_cgm_encoders/results/gru_tuning_trials.csv
default config ranked #6 of 40 by mean_val_loss


## Final confirmatory evaluation

In [ ]:
best_params = study.best_params
best_train_loader = load_tensors('train', batch_size=best_params['batch_size'])

winner_seed_results = []
for seed in FINAL_SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")
    train.set_seed(seed)
    model = GRUPredictor(hidden_size=best_params['hidden_size'],
                          num_layers=best_params['num_layers'],
                          dropout=best_params['dropout'])
    model, history = train.train_model(
        model, best_train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=best_params['lr'], weight_decay=best_params['weight_decay'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / f'winner_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'winner/seed={seed}',
    )

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    winner_seed_results.append({
        'seed': seed, 'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    })

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")


----------------seed=7----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


winner/seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

seed=  7  test RMSE @ [15, 30, 60, 90, 120] min = [11.14, 18.39, 27.53, 33.19, 36.25]

----------------seed=14----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


winner/seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 14  test RMSE @ [15, 30, 60, 90, 120] min = [11.15, 18.44, 27.52, 32.98, 36.01]

----------------seed=21----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


...

seed= 56  test RMSE @ [15, 30, 60, 90, 120] min = [11.16, 18.63, 27.91, 33.47, 36.43]

----------------seed=63----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


winner/seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 63  test RMSE @ [15, 30, 60, 90, 120] min = [11.4, 18.69, 27.68, 33.1, 36.14]

----------------seed=70----------------
device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


winner/seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

seed= 70  test RMSE @ [15, 30, 60, 90, 120] min = [11.18, 18.39, 27.47, 33.03, 36.13]


In [ ]:
# Same two-way variance decomposition as every other backbone-screen model: sd1 across
# FINAL_SEEDS, sd2 across test participants (computed independently, never folded into the mean).
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}
for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [r['test_results'][h]['ceg']['A'] for r in winner_seed_results]
        elif m == 'zone_ab':
            seed_values = [r['test_results'][h]['ceg']['A'] + r['test_results'][h]['ceg']['B']
                            for r in winner_seed_results]
        else:
            seed_values = [r['test_results'][h][m] for r in winner_seed_results]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [_extract(r['test_results_per_participant'][pid][h], m)
                                    for r in winner_seed_results]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))

        summary[h][m] = {'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2}

print(f'=== {MODEL_NAME} test set: overall mean +/- SD_seed (SD_participant) ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

with open(FINAL_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'best_params': best_params,
        'search_seed': SEARCH_SEED,
        'trial_seeds': TRIAL_SEEDS,
        'n_trials': N_TRIALS,
        'final_seeds': FINAL_SEEDS,
        'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved to {FINAL_SUMMARY_PATH}')

=== GRU-tuned test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.25 +/- 0.10 (1.92) mg/dL   MAE = 7.64 +/- 0.10 (1.49)   gRMSE = 12.46 +/- 0.23 (3.06)   Zone A = 97.11 +/- 0.11 (2.07)%   Zone A+B = 99.77 +/- 0.02 (0.37)%
 30-min:  RMSE = 18.57 +/- 0.13 (3.93) mg/dL   MAE = 12.34 +/- 0.12 (2.70)   gRMSE = 21.57 +/- 0.31 (6.53)   Zone A = 90.52 +/- 0.16 (3.20)%   Zone A+B = 99.52 +/- 0.03 (0.62)%
 60-min:  RMSE = 27.70 +/- 0.18 (9.40) mg/dL   MAE = 18.30 +/- 0.14 (5.98)   gRMSE = 33.91 +/- 0.36 (14.73)   Zone A = 81.23 +/- 0.33 (5.15)%   Zone A+B = 98.90 +/- 0.05 (1.13)%
 90-min:  RMSE = 33.23 +/- 0.21 (13.40) mg/dL   MAE = 22.13 +/- 0.17 (8.37)   gRMSE = 41.53 +/- 0.37 (20.76)   Zone A = 74.82 +/- 0.42 (7.03)%   Zone A+B = 98.26 +/- 0.06 (1.85)%
120-min:  RMSE = 36.25 +/- 0.18 (15.85) mg/dL   MAE = 24.39 +/- 0.14 (10.19)   gRMSE = 45.71 +/- 0.31 (24.45)   Zone A = 71.02 +/- 0.42 (7.60)%   Zone A+B = 97.88 +/- 0.07 (2.82)%

saved to /content/drive/MyDrive/dissert

In [ ]:
# Direct, same-protocol comparison against the existing untuned-default GRU stability summary
# (already computed in experiments/1_baseline; not rerun here).
with open(DEFAULT_SUMMARY_PATH) as f:
    default_summary = json.load(f)['per_horizon_summary']

print(f"{'Horizon':>9} | {'Default RMSE':>22} | {'Tuned RMSE':>22} | {'Diff':>7}")
print('-' * 68)
for h in HORIZONS:
    d = default_summary[str(h)]['rmse']
    t = summary[h]['rmse']
    diff = t['mean'] - d['mean']
    print(f"{h:>6} min | "
          f"{d['mean']:6.2f} +/- {d['sd1_across_seeds']:.2f} ({d['sd2_across_participants']:5.2f}) | "
          f"{t['mean']:6.2f} +/- {t['sd1_across_seeds']:.2f} ({t['sd2_across_participants']:5.2f}) | "
          f"{diff:+.2f}")

  Horizon |           Default RMSE |             Tuned RMSE |    Diff
--------------------------------------------------------------------
    15 min |  11.29 +/- 0.12 ( 2.01) |  11.25 +/- 0.10 ( 1.92) | -0.04
    30 min |  18.61 +/- 0.10 ( 3.99) |  18.57 +/- 0.13 ( 3.93) | -0.04
    60 min |  27.59 +/- 0.13 ( 9.24) |  27.70 +/- 0.18 ( 9.40) | +0.10
    90 min |  32.97 +/- 0.23 (12.97) |  33.23 +/- 0.21 (13.40) | +0.26
   120 min |  36.11 +/- 0.30 (15.39) |  36.25 +/- 0.18 (15.85) | +0.13
